# Import Module & Library

In [28]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer



## Load Dataset

In [3]:
data_path = os.path.join("../../Data/Telco_Customer_Churn_Cleaned.csv")

try:
    df = pd.read_csv(data_path, sep=';')
    print(f"Data Shape: {df.shape}")
except Exception as e:
    print(f"Error: {e}")

df.head()

Data Shape: (7032, 20)


,customerID,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,FamilyStatus
0,7590-VHVEG,Female,0,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,Couple
1,5575-GNVDE,Male,0,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,Single
2,3668-QPYBK,Male,0,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Single
3,7795-CFOCW,Male,0,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,Single
4,9237-HQITU,Female,0,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Single


# Data Transformation and Standarization

## Drop unnecesary feature 

In [8]:
df_work = df.copy()
drop_cols = ["customerID", "gender", "TotalCharges"]
drop_cols = [c for c in drop_cols if c in df_work.columns]
df_work = df_work.drop(columns=drop_cols)

## Copy and Drop curn

In [11]:
if "Churn" in df_work.columns:
    y = df_work["Churn"].copy()
    X = df_work.drop(columns=["Churn"])
else:
    X = df_work.copy()

## Transfroming tenure data

In [12]:
for c in X.select_dtypes(include="object").columns:
    X[c] = X[c].astype(str).str.strip()

def tenure_to_segment(col):
    if isinstance(col, pd.DataFrame):
        s = col.iloc[:, 0]
    else:
        s = pd.Series(col.ravel())

    s = pd.to_numeric(s, errors="coerce").fillna(0)

    bins = [-np.inf, 12, 24, 48, np.inf]
    labels = ['New Customer', "Regular Customer", "Loyal Customer", "Very Loyal"]

    seg = pd.cut(s, bins=bins, labels=labels, include_lowest=True)
    return pd.DataFrame({"tenure": seg.astype(str)})

C:\Users\GF63\AppData\Local\Temp\ipykernel_3868\2025098806.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in X.select_dtypes(include="object").columns:


## Build transform & encode pipe

In [25]:
# Normalisasi nilai layanan agar kolom biner tetap benar-benar Yes/No
replace_no_service_cols = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in replace_no_service_cols:
    if col in X.columns:
        X[col] = X[col].replace({
            "No internet service": "No",
            "No phone service": "No"
        })

# Tenure -> segment -> ordinal
tenure_pipe = Pipeline(steps=[
    ("bucket", FunctionTransformer(
        tenure_to_segment,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("ord", OrdinalEncoder(
        categories=[["New Customer", "Regular Customer", "Loyal Customer", "Very Loyal"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# Contract ordinal
contract_pipe = OrdinalEncoder(
    categories=[["Month-to-month", "One year", "Two year"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Pisahkan kolom kategorikal: biner vs non-biner
cat_cols = X.select_dtypes(include="object").columns.tolist()

binary_yesno_cols = []
multiclass_nominal_cols = []

for col in cat_cols:
    if col == "Contract":
        continue
    uniq = set(X[col].dropna().astype(str).str.strip().str.lower().unique())
    if uniq.issubset({"yes", "no"}):
        binary_yesno_cols.append(col)
    else:
        multiclass_nominal_cols.append(col)

print("Binary map (Yes/No -> 0/1):", binary_yesno_cols)
print("OHE multiclass:", multiclass_nominal_cols)

# Mapper untuk kolom biner (tetap 1 kolom per fitur)
def map_yes_no_block(df_part):
    out = df_part.copy()
    for c in out.columns:
        out[c] = (
            out[c]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"no": 0.0, "yes": 1.0})
        )
    return out

binary_pipe = Pipeline(steps=[
    ("yn_map", FunctionTransformer(
        map_yes_no_block,
        validate=False,
        feature_names_out="one-to-one"
    ))
])

# Numeric selain tenure
num_cols = X.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != "tenure"]

# Build ColumnTransformer dinamis
transformers = [
    ("tenure_ord", tenure_pipe, ["tenure"]),
    ("contract_ord", contract_pipe, ["Contract"]),
]

if binary_yesno_cols:
    transformers.append(("bin_map", binary_pipe, binary_yesno_cols))

if multiclass_nominal_cols:
    transformers.append((
        "cat_ohe",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        multiclass_nominal_cols
    ))

if num_cols:
    transformers.append(("num", "passthrough", num_cols))

transform_pipeline = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)



Binary map (Yes/No -> 0/1): ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']
OHE multiclass: ['InternetService', 'PaymentMethod', 'FamilyStatus']


C:\Users\GF63\AppData\Local\Temp\ipykernel_3868\3363484652.py:35: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


## Encoding results

In [26]:
X_encoded = transform_pipeline.fit_transform(X)
encoded_cols = transform_pipeline.get_feature_names_out()
X_encoded_df = pd.DataFrame(X_encoded, columns=encoded_cols, index=X.index)

X_encoded_df.head(10)


,tenure,Contract,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,FamilyStatus_Couple,FamilyStatus_Family,FamilyStatus_Single,FamilyStatus_Single Parent,SeniorCitizen,MonthlyCharges
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,29.85
1,2.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,56.95
2,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,53.85
3,2.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,42.30
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,70.70
5,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,99.65
6,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,89.10
7,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,29.75
8,2.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,104.80
9,3.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,56.15


# Feature Selections

## Split the data

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Define the feature selection params

In [29]:
l1_selector_estimator = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    max_iter=5000,
    random_state=42
)

## Full selection pipeline

In [30]:
svc_fs_pipeline = Pipeline(steps=[
    ("prep", transform_pipeline),
    ("scale", StandardScaler()),
    ("fs", SelectFromModel(estimator=l1_selector_estimator, threshold="median")),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42))
])

svc_fs_pipeline.fit(X_train, y_train)

,steps,"[('prep', ...), ('scale', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tenure_ord', ...), ('contract_ord', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [31]:
feature_names = svc_fs_pipeline.named_steps["prep"].get_feature_names_out()
mask = svc_fs_pipeline.named_steps["fs"].get_support()

selected_features = pd.Index(feature_names)[mask]
print("Jumlah fitur sebelum:", len(feature_names))
print("Jumlah fitur terpilih:", len(selected_features))
selected_features

Jumlah fitur sebelum: 24
Jumlah fitur terpilih: 12


Index(['tenure', 'Contract', 'PhoneService', 'MultipleLines', 'OnlineSecurity',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
       'InternetService_Fiber optic', 'InternetService_No',
       'PaymentMethod_Electronic check'],
      dtype='str')